In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS = Path("../results")

CONFIRMATORY_DIR = RESULTS / "week12" / "confirmatory"
HARMONIZED_DIR = RESULTS / "week12" / "harmonized_n100"

In [2]:
confirmatory = pd.read_csv(
    CONFIRMATORY_DIR / "primary_power_runs.csv",
    keep_default_na=False,
    na_values=[""]
)

confirmatory.shape

(22000, 18)

In [3]:
confirmatory["N"].value_counts().sort_index()

N
50     12000
200    10000
Name: count, dtype: int64

In [4]:
harmonized_n100 = pd.read_csv(
    HARMONIZED_DIR / "harmonized_n100_power_runs.csv",
    keep_default_na=False,
    na_values=[""]
)

harmonized_n100.shape

(15000, 18)

In [5]:
harmonized_n100["N"].value_counts()

N
100    15000
Name: count, dtype: int64

In [6]:
confirmatory = confirmatory.copy()
harmonized_n100 = harmonized_n100.copy()

confirmatory["source_status"] = "confirmatory"
harmonized_n100["source_status"] = "posthoc_harmonized_n100"

In [7]:
confirmatory["analysis_role"] = "primary_confirmatory_extension"
harmonized_n100["analysis_role"] = "harmonized_sample_size_comparison"

In [8]:
combined_runs = pd.concat(
    [
        confirmatory,
        harmonized_n100
    ],
    ignore_index=True
)

In [9]:
combined_runs.shape

(37000, 20)

In [10]:
combined_runs.groupby(["source_status", "N"]).size()

source_status            N  
confirmatory             50     12000
                         200    10000
posthoc_harmonized_n100  100    15000
dtype: int64

In [11]:
cell_cols = [
    "scenario",
    "severity",
    "test",
    "N"
]

n_cells = (
    combined_runs[cell_cols]
    .drop_duplicates()
    .shape[0]
)

print(n_cells)

185


In [12]:
cell_summary = (
    combined_runs
    .groupby(
        [
            "scenario",
            "severity",
            "N",
            "test",
            "source_status"
        ],
        as_index=False
    )
    .agg(
        rejections=("reject", "sum"),
        repetitions=("reject", "size")
    )
)

In [13]:
cell_summary["power"] = (
    cell_summary["rejections"]
    / cell_summary["repetitions"]
)

In [14]:
cell_summary.shape

(185, 8)

In [15]:
cell_summary["repetitions"].value_counts()

repetitions
200    185
Name: count, dtype: int64

In [16]:
n100_check = cell_summary[
    cell_summary["N"] == 100
]

print(len(n100_check))
print(n100_check["source_status"].value_counts())

75
source_status
posthoc_harmonized_n100    75
Name: count, dtype: int64


In [17]:
OUT = RESULTS / "week12" / "harmonized_analysis"
OUT.mkdir(parents=True, exist_ok=True)

combined_runs.to_csv(
    OUT / "combined_power_runs_harmonized_n100.csv",
    index=False
)

cell_summary.to_csv(
    OUT / "combined_power_cells_harmonized_n100.csv",
    index=False
)